# DisputeAI – Member 2: PDF Extraction, RAG and Investigation


**Scope and evidence note:** This notebook processes the two RBI PDFs currently present in `documents/rbi`. The 2017 Customer Protection circular is marked historical/withdrawn and must not be presented as current law. The 2026 Integrated Ombudsman Scheme concerns complaint redressal; it does not supply current customer-liability rules. The `documents/bank_policies` folder is currently empty, so bank-specific policy evidence is not available. Fraud classification is rule-based and its reported test accuracy applies only to the small manually prepared test set.


In [1]:
%pip install pypdf

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
from pypdf import PdfReader
import json

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
from pathlib import Path

# Correct project folder paths
RBI_FOLDER = Path("documents/rbi")
PROCESSED_FOLDER = Path("documents/processed")
METADATA_FOLDER = Path("documents/metadata")

PROCESSED_FOLDER.mkdir(parents=True, exist_ok=True)
METADATA_FOLDER.mkdir(parents=True, exist_ok=True)

print("Project paths configured successfully!")

Project paths configured successfully!


In [4]:
# Find all PDF files in the RBI folder
pdf_files = list(RBI_FOLDER.glob("*.pdf"))

print("PDF documents found:", len(pdf_files))

for pdf in pdf_files:
    print("File:", pdf.name)

PDF documents found: 2
File: RBI_Customer_Protection_2017.pdf
File: RBI_Integrated_Ombudsman_Scheme_2026.pdf


In [5]:
# Store extracted text from both documents
extracted_documents = {}

for pdf_path in pdf_files:

    reader = PdfReader(str(pdf_path))
    pages = []

    print(f"\nProcessing: {pdf_path.name}")

    for page_number, page in enumerate(reader.pages, start=1):

        page_text = page.extract_text() or ""

        pages.append({
            "page_number": page_number,
            "text": page_text.strip()
        })

    extracted_documents[pdf_path.name] = pages

    print("Total pages:", len(pages))
    print("Pages with text:", sum(1 for p in pages if p["text"]))
    print("Pages without text:", sum(1 for p in pages if not p["text"]))

print("\nPDF extraction completed!")


Processing: RBI_Customer_Protection_2017.pdf
Total pages: 9
Pages with text: 9
Pages without text: 0

Processing: RBI_Integrated_Ombudsman_Scheme_2026.pdf
Total pages: 24
Pages with text: 24
Pages without text: 0

PDF extraction completed!


In [6]:
for filename, pages in extracted_documents.items():

    print("\nDOCUMENT:", filename)

    for page in pages[:2]:
        print(f"\n--- PAGE {page['page_number']} ---")
        print(page["text"][:1000])


DOCUMENT: RBI_Customer_Protection_2017.pdf

--- PAGE 1 ---
____________________________________________________________________________________ 
ब��कं ग�व�नयमन�वभाग,क� द्र�यकायार्लय, 12वींऔर 13वींमंिज़ल, क� द्र�यकायार्लयभवन, शह�दभगत�संहमागर्, मुंबई 400001 
टेल�फोन /Tel No: 22661602, 22601000 फै क् स /Fax No: 022-2270 5670, 2260 5671, 5691 2270, 2260 5692 
Department of Banking Regulation,Central Office, 12th & 13th Floor, Central Office Bhavan, ShahidBhagat Singh Marg, Mumbai - 400001 
Tel No: 22661602, 22601000  Fax No: 022-2270 5670, 2260 5671, 5691 2270, 2260 5692 
RBI/2017-18/15 
DBR.No.Leg.BC.78/09.07.005/2017-18  July 6, 2017 
A
ll Scheduled Commercial Banks (including RRBs) 
All Small Finance Banks and Payments Banks 
De
ar Sir/ Madam, 
Customer Protection – Limiting Liability of Customers in Unauthorised 
Electronic Banking Transactions  
P
lease refer to our circular DBOD.Leg.BC.86/09.07.007/2001- 02 dated April 8, 2002 
regarding reversal of erroneous debits arising from frau

In [7]:
for filename, pages in extracted_documents.items():

    output_name = Path(filename).stem + "_extracted.txt"
    output_path = PROCESSED_FOLDER / output_name

    with open(output_path, "w", encoding="utf-8") as file:

        for page in pages:
            file.write(f"\n\n--- PAGE {page['page_number']} ---\n")
            file.write(page["text"])
            file.write("\n")

    print("Saved successfully:", output_name)

Saved successfully: RBI_Customer_Protection_2017_extracted.txt
Saved successfully: RBI_Integrated_Ombudsman_Scheme_2026_extracted.txt


In [8]:
json_output = []

for filename, pages in extracted_documents.items():

    for page in pages:
        json_output.append({
            "source": filename,
            "page_number": page["page_number"],
            "text": page["text"]
        })

json_path = METADATA_FOLDER / "rbi_extracted_pages.json"

with open(json_path, "w", encoding="utf-8") as file:
    json.dump(json_output, file, ensure_ascii=False, indent=4)

print("JSON saved successfully:", json_path)
print("Total page records:", len(json_output))

JSON saved successfully: documents\metadata\rbi_extracted_pages.json
Total page records: 33


In [9]:
print("Processed documents:")

for file in PROCESSED_FOLDER.iterdir():
    print(file.name)

print("\nMetadata documents:")

for file in METADATA_FOLDER.iterdir():
    print(file.name)

Processed documents:
rbi_chunks.json
rbi_context_chunks.json
RBI_Customer_Protection_2017_cleaned.txt
RBI_Customer_Protection_2017_extracted.txt
RBI_Integrated_Ombudsman_Scheme_2026_cleaned.txt
RBI_Integrated_Ombudsman_Scheme_2026_extracted.txt
rbi_liability_context.json
rbi_table_chunks.json
rbi_verified_table_chunks.json

Metadata documents:
rbi_cleaned_pages.json
rbi_extracted_pages.json


## 1. Text cleaning and preprocessing


In [10]:
import re

def clean_page_text(text):
    # Normalize line endings
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    # Remove excessive spaces and tabs
    text = re.sub(r"[ \t]+", " ", text)

    # Remove excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove spaces at line beginnings and endings
    text = "\n".join(
        line.strip() for line in text.split("\n")
    )

    return text.strip()


print("Text cleaning function created successfully!")

Text cleaning function created successfully!


In [11]:
cleaned_documents = {}

for filename, pages in extracted_documents.items():

    cleaned_pages = []

    for page in pages:

        cleaned_pages.append({
            "page_number": page["page_number"],
            "text": clean_page_text(page["text"])
        })

    cleaned_documents[filename] = cleaned_pages

    print(
        filename,
        "->",
        len(cleaned_pages),
        "pages cleaned"
    )

print("\nPreprocessing completed!")

RBI_Customer_Protection_2017.pdf -> 9 pages cleaned
RBI_Integrated_Ombudsman_Scheme_2026.pdf -> 24 pages cleaned

Preprocessing completed!


In [12]:
for filename, pages in cleaned_documents.items():

    print("\nDOCUMENT:", filename)
    print("\nPAGE 1:")
    print(pages[0]["text"][:1200])


DOCUMENT: RBI_Customer_Protection_2017.pdf

PAGE 1:
____________________________________________________________________________________
ब��कं ग�व�नयमन�वभाग,क� द्र�यकायार्लय, 12वींऔर 13वींमंिज़ल, क� द्र�यकायार्लयभवन, शह�दभगत�संहमागर्, मुंबई 400001
टेल�फोन /Tel No: 22661602, 22601000 फै क् स /Fax No: 022-2270 5670, 2260 5671, 5691 2270, 2260 5692
Department of Banking Regulation,Central Office, 12th & 13th Floor, Central Office Bhavan, ShahidBhagat Singh Marg, Mumbai - 400001
Tel No: 22661602, 22601000 Fax No: 022-2270 5670, 2260 5671, 5691 2270, 2260 5692
RBI/2017-18/15
DBR.No.Leg.BC.78/09.07.005/2017-18 July 6, 2017
A
ll Scheduled Commercial Banks (including RRBs)
All Small Finance Banks and Payments Banks
De
ar Sir/ Madam,
Customer Protection – Limiting Liability of Customers in Unauthorised
Electronic Banking Transactions
P
lease refer to our circular DBOD.Leg.BC.86/09.07.007/2001- 02 dated April 8, 2002
regarding reversal of erroneous debits arising from fraudulent or other transac

In [13]:
for filename, pages in cleaned_documents.items():

    output_name = Path(filename).stem + "_cleaned.txt"
    output_path = PROCESSED_FOLDER / output_name

    with open(output_path, "w", encoding="utf-8") as file:

        for page in pages:
            file.write(f"\n\n--- PAGE {page['page_number']} ---\n")
            file.write(page["text"])
            file.write("\n")

    print("Saved:", output_name)

print("\nCleaned documents saved successfully!")

Saved: RBI_Customer_Protection_2017_cleaned.txt
Saved: RBI_Integrated_Ombudsman_Scheme_2026_cleaned.txt

Cleaned documents saved successfully!


In [14]:
cleaned_records = []

for filename, pages in cleaned_documents.items():

    for page in pages:
        cleaned_records.append({
            "source": filename,
            "page_number": page["page_number"],
            "text": page["text"],
            "extraction_status": "needs_quality_review"
                if filename == "RBI_Customer_Protection_2017.pdf"
                else "extracted_check_completed"
        })

cleaned_json_path = METADATA_FOLDER / "rbi_cleaned_pages.json"

with open(cleaned_json_path, "w", encoding="utf-8") as file:
    json.dump(cleaned_records, file, ensure_ascii=False, indent=4)

print("Cleaned metadata saved!")
print("Total page records:", len(cleaned_records))

Cleaned metadata saved!
Total page records: 33


In [15]:
print("PROCESSED FILES")

for file in PROCESSED_FOLDER.iterdir():
    print(file.name)

print("\nMETADATA FILES")

for file in METADATA_FOLDER.iterdir():
    print(file.name)

PROCESSED FILES
rbi_chunks.json
rbi_context_chunks.json
RBI_Customer_Protection_2017_cleaned.txt
RBI_Customer_Protection_2017_extracted.txt
RBI_Integrated_Ombudsman_Scheme_2026_cleaned.txt
RBI_Integrated_Ombudsman_Scheme_2026_extracted.txt
rbi_liability_context.json
rbi_table_chunks.json
rbi_verified_table_chunks.json

METADATA FILES
rbi_cleaned_pages.json
rbi_extracted_pages.json


In [16]:
# Inspect suspicious broken words in the 2017 document

for page in cleaned_documents["RBI_Customer_Protection_2017.pdf"]:

    suspicious_lines = []

    for line in page["text"].splitlines():

        if any(word in line for word in ["A ll", "De ar", "W ith", "unauthor ised"]):
            suspicious_lines.append(line)

    if suspicious_lines:
        print(f"\nPAGE {page['page_number']}")

        for line in suspicious_lines:
            print(line)


PAGE 1
considering the recent surge in customer grievances relating to unauthor ised


In [17]:
for page in cleaned_documents["RBI_Customer_Protection_2017.pdf"]:

    page["text"] = page["text"].replace(
        "unauthor ised",
        "unauthorised"
    )

print("Verified spacing correction completed!")

Verified spacing correction completed!


## 2. Chunking and saving processed documents


In [18]:
def create_chunks(text, chunk_size=1000, overlap=200):

    words = text.split()

    chunks = []
    start = 0

    while start < len(words):

        end = start + chunk_size

        chunk_text = " ".join(words[start:end])

        chunks.append(chunk_text)

        start += chunk_size - overlap

    return chunks


print("Chunking function created successfully!")

Chunking function created successfully!


In [19]:
# Store all generated chunks
all_chunks = []

for filename, pages in cleaned_documents.items():

    for page in pages:

        page_text = page["text"]

        # Split each page into smaller chunks
        page_chunks = create_chunks(
            page_text,
            chunk_size=1000,
            overlap=200
        )

        for index, chunk_text in enumerate(page_chunks):

            if chunk_text.strip():

                all_chunks.append({
                    "chunk_id": f"{Path(filename).stem}_P{page['page_number']}_C{index + 1}",
                    "source": filename,
                    "page_number": page["page_number"],
                    "chunk_index": index + 1,
                    "text": chunk_text
                })

print("Chunking completed successfully!")
print("Total chunks created:", len(all_chunks))

Chunking completed successfully!
Total chunks created: 33


In [20]:
for chunk in all_chunks[:3]:

    print("\nChunk ID:", chunk["chunk_id"])
    print("Source:", chunk["source"])
    print("Page:", chunk["page_number"])
    print("Text:")
    print(chunk["text"][:500])
    print("-" * 60)


Chunk ID: RBI_Customer_Protection_2017_P1_C1
Source: RBI_Customer_Protection_2017.pdf
Page: 1
Text:
____________________________________________________________________________________ ब��कं ग�व�नयमन�वभाग,क� द्र�यकायार्लय, 12वींऔर 13वींमंिज़ल, क� द्र�यकायार्लयभवन, शह�दभगत�संहमागर्, मुंबई 400001 टेल�फोन /Tel No: 22661602, 22601000 फै क् स /Fax No: 022-2270 5670, 2260 5671, 5691 2270, 2260 5692 Department of Banking Regulation,Central Office, 12th & 13th Floor, Central Office Bhavan, ShahidBhagat Singh Marg, Mumbai - 400001 Tel No: 22661602, 22601000 Fax No: 022-2270 5670, 2260 5671, 5691 2270, 2
------------------------------------------------------------

Chunk ID: RBI_Customer_Protection_2017_P2_C1
Source: RBI_Customer_Protection_2017.pdf
Page: 2
Text:
RBI,DBR,CO - 2 - continuation sheet 4. The systems and procedures in banks must be designed to make customers feel safe about carrying out electronic banking transactions. To achieve this, banks must put in place: (i) appropriate system

In [21]:
# Save all chunks in JSON format

chunks_json_path = PROCESSED_FOLDER / "rbi_chunks.json"

with open(chunks_json_path, "w", encoding="utf-8") as file:
    json.dump(all_chunks, file, ensure_ascii=False, indent=4)

print("Chunks saved successfully!")
print("File:", chunks_json_path)
print("Total chunks saved:", len(all_chunks))

Chunks saved successfully!
File: documents\processed\rbi_chunks.json
Total chunks saved: 33


In [22]:
from collections import Counter

chunk_counts = Counter(
    chunk["source"] for chunk in all_chunks
)

for filename, count in chunk_counts.items():
    print(filename, ":", count, "chunks")

print("\nTotal chunks:", sum(chunk_counts.values()))

RBI_Customer_Protection_2017.pdf : 9 chunks
RBI_Integrated_Ombudsman_Scheme_2026.pdf : 24 chunks

Total chunks: 33


## 3. Embeddings and ChromaDB


In [23]:
%pip install chromadb sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
import chromadb
import sentence_transformers

print("ChromaDB version:", chromadb.__version__)
print("Sentence Transformers version:", sentence_transformers.__version__)

print("\nBoth packages imported successfully!")

C:\Users\GOD\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChromaDB version: 1.5.9
Sentence Transformers version: 6.2.0

Both packages imported successfully!


In [25]:
import json
from pathlib import Path

chunks_json_path = Path("documents/processed/rbi_chunks.json")

with open(chunks_json_path, "r", encoding="utf-8") as file:
    all_chunks = json.load(file)

print("Chunks loaded successfully!")
print("Total chunks:", len(all_chunks))
print("First chunk ID:", all_chunks[0]["chunk_id"])

Chunks loaded successfully!
Total chunks: 33
First chunk ID: RBI_Customer_Protection_2017_P1_C1


In [26]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1621.93it/s]


Embedding model loaded successfully!


In [27]:
# Extract text from all chunks
chunk_texts = [chunk["text"] for chunk in all_chunks]

# Generate embeddings
embeddings = model.encode(
    chunk_texts,
    show_progress_bar=True
)

print("Embeddings generated successfully!")
print("Total embeddings:", len(embeddings))
print("Vector dimensions:", embeddings.shape[1])

Batches: 100%|██████████| 2/2 [00:02<00:00,  1.22s/it]

Embeddings generated successfully!
Total embeddings: 33
Vector dimensions: 384


In [28]:
import chromadb

# Create a folder for the vector database
CHROMA_FOLDER = "documents/chroma_db"

# Create persistent ChromaDB client
client = chromadb.PersistentClient(path=CHROMA_FOLDER)

# Create a collection
collection = client.get_or_create_collection(
    name="rbi_regulations",
    metadata={"description": "RBI regulatory documents for DisputeAI"}
)

print("ChromaDB database created successfully!")
print("Collection name:", collection.name)

ChromaDB database created successfully!
Collection name: rbi_regulations


In [29]:
# Prepare unique IDs for all chunks
chunk_ids = [chunk["chunk_id"] for chunk in all_chunks]

# Prepare metadata for each chunk
chunk_metadatas = [
    {
        "source": chunk["source"],
        "page_number": chunk["page_number"],
        "chunk_index": chunk["chunk_index"]
    }
    for chunk in all_chunks
]

# Store documents, embeddings and metadata
collection.upsert(
    ids=chunk_ids,
    documents=chunk_texts,
    embeddings=embeddings.tolist(),
    metadatas=chunk_metadatas
)

print("All chunks stored in ChromaDB!")
print("Total records:", collection.count())

All chunks stored in ChromaDB!
Total records: 36


In [30]:
query = """
A customer reports an unauthorised electronic banking
transaction three weeks after it happened and shared
an OTP with a caller impersonating the bank.
What RBI rules discuss customer liability?
"""

# Convert the question into an embedding
query_embedding = model.encode(query).tolist()

# Search ChromaDB
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)

# Display retrieved results
for i, document in enumerate(results["documents"][0]):
    metadata = results["metadatas"][0][i]

    print("\nResult:", i + 1)
    print("Source:", metadata["source"])
    print("Page:", metadata["page_number"])
    print("Relevant text:")
    print(document[:700])
    print("-" * 60)


Result: 1
Source: RBI_Customer_Protection_2017.pdf
Page: 3
Relevant text:
RBI,DBR,CO - 3 - continuation sheet communication systems used by banks to send alerts and receive their responses thereto must record the time and date of delivery of the message and receipt of customer’s response, if any, to them. This shall be important in determining the extent of a customer’s liability. The banks may not offer facility of electronic transactions, other than ATM cash withdrawals, to customers who do not provide mobile numbers to the bank. On receipt of report of an unauthorised transaction from the customer, banks must take immediate steps to prevent further unauthorised transactions in the account. Limited Liability of a Customer (a) Zero Liability of a Customer 6. A cu
------------------------------------------------------------

Result: 2
Source: RBI_Customer_Protection_2017.pdf
Page: 6
Relevant text:
RBI,DBR,CO - 6 - continuation sheet Board Approved Policy for Customer Protection 11. Ta

In [31]:
query_2026 = """
How can a customer file a complaint against a bank
under the RBI Integrated Ombudsman Scheme 2026?
What are the eligibility and maintainability requirements?
"""

query_embedding_2026 = model.encode(query_2026).tolist()

results_2026 = collection.query(
    query_embeddings=[query_embedding_2026],
    n_results=5,
    where={"source": "RBI_Integrated_Ombudsman_Scheme_2026.pdf"}
)

for i, document in enumerate(results_2026["documents"][0]):
    metadata = results_2026["metadatas"][0][i]

    print("\nResult:", i + 1)
    print("Page:", metadata["page_number"])
    print(document[:700])
    print("-" * 60)


Result: 1
Page: 22
22 (iv) Whether your complaint has already been dealt with by the RBI Ombudsman earlier or is under process on the same ground/ cause of action with the RBI Ombudsman? Yes No (v) Is the Complainant a staff of the RE and complaint involves employer-employee relationship? Yes No Note: If you answered “Yes” to (i), (iv) or (v), or if you answered “Yes” to (ii) and “No” to (iii), your complaint is not maintainable under the Scheme and will not be registered. g) What is the nature of your complaint (please select one – place a tick mark in the box) - (The response to this question is optional for the Complainant) i. ATM/Debit Cards ii. Bank Guarantees/ Letter of Credit and documentary credits iii
------------------------------------------------------------

Result: 2
Page: 18
18 Annex Complaint Form Form for filing of online and offline complaints with the RBI Ombudsman Part A: Guidance for filing of a complaint The salient mandatory requirements under the captioned sche

In [32]:
# Add document status to existing ChromaDB records

records = collection.get(include=["metadatas"])

for record_id, metadata in zip(records["ids"], records["metadatas"]):

    source = metadata["source"]

    if source == "RBI_Customer_Protection_2017.pdf":
        metadata["document_status"] = "historical"
        metadata["document_scope"] = "customer_liability"

    elif source == "RBI_Integrated_Ombudsman_Scheme_2026.pdf":
        metadata["document_status"] = "effective_from_2026"
        metadata["document_scope"] = "complaint_redressal"

    collection.update(
        ids=[record_id],
        metadatas=[metadata]
    )

print("Document metadata updated successfully!")
print("Total records:", collection.count())

Document metadata updated successfully!
Total records: 36


## 4. Preserve important RBI table evidence


In [33]:
import json
from pathlib import Path

with open(
    "documents/processed/rbi_chunks.json",
    "r",
    encoding="utf-8"
) as file:
    chunks = json.load(file)

print("TOTAL CHUNKS:", len(chunks))

for chunk in chunks[:10]:
    print("\nChunk ID:", chunk["chunk_id"])
    print("Source:", chunk["source"])
    print("Page:", chunk["page_number"])
    print("Text length:", len(chunk["text"]))
    print("Text preview:", chunk["text"][:200])
    print("-" * 60)

TOTAL CHUNKS: 33

Chunk ID: RBI_Customer_Protection_2017_P1_C1
Source: RBI_Customer_Protection_2017.pdf
Page: 1
Text length: 1926
Text preview: ____________________________________________________________________________________ ब��कं ग�व�नयमन�वभाग,क� द्र�यकायार्लय, 12वींऔर 13वींमंिज़ल, क� द्र�यकायार्लयभवन, शह�दभगत�संहमागर्, मुंबई 400001 टेल�फ
------------------------------------------------------------

Chunk ID: RBI_Customer_Protection_2017_P2_C1
Source: RBI_Customer_Protection_2017.pdf
Page: 2
Text length: 2421
Text preview: RBI,DBR,CO - 2 - continuation sheet 4. The systems and procedures in banks must be designed to make customers feel safe about carrying out electronic banking transactions. To achieve this, banks must 
------------------------------------------------------------

Chunk ID: RBI_Customer_Protection_2017_P3_C1
Source: RBI_Customer_Protection_2017.pdf
Page: 3
Text length: 1997
Text preview: RBI,DBR,CO - 3 - continuation sheet communication systems used by banks to sen

In [34]:
for page_number in [3, 4, 5, 6]:

    page_data = [
        chunk for chunk in chunks
        if chunk["page_number"] == page_number
        and chunk["source"] == "RBI_Customer_Protection_2017.pdf"
    ]

    for item in page_data:
        print("\n" + "=" * 70)
        print("PAGE:", page_number)
        print("=" * 70)
        print(item["text"])


PAGE: 3
RBI,DBR,CO - 3 - continuation sheet communication systems used by banks to send alerts and receive their responses thereto must record the time and date of delivery of the message and receipt of customer’s response, if any, to them. This shall be important in determining the extent of a customer’s liability. The banks may not offer facility of electronic transactions, other than ATM cash withdrawals, to customers who do not provide mobile numbers to the bank. On receipt of report of an unauthorised transaction from the customer, banks must take immediate steps to prevent further unauthorised transactions in the account. Limited Liability of a Customer (a) Zero Liability of a Customer 6. A customer’s entitlement to zero liability shall arise where the unauthorised transaction occurs in the following events: (i) Contributory fraud/ negligence/ deficiency on the part of the bank (irrespective of whether or not the transaction is reported by the customer). (ii) Third party breach 

In [35]:

import json
from pathlib import Path

# Select connected pages from the historical RBI circular
selected_pages = [
    chunk for chunk in chunks
    if chunk["source"] == "RBI_Customer_Protection_2017.pdf"
    and chunk["page_number"] in [3, 4, 5, 6]
]

selected_pages.sort(key=lambda x: x["page_number"])

# Combine pages while retaining page markers
combined_text = ""

for item in selected_pages:
    combined_text += (
        f"\n[PAGE {item['page_number']}]\n"
        + item["text"]
        + "\n"
    )

# Create overlapping chunks
chunk_size = 1200
overlap = 250
improved_chunks = []

for start in range(
    0,
    len(combined_text),
    chunk_size - overlap
):
    end = min(start + chunk_size, len(combined_text))
    chunk_text = combined_text[start:end]

    # Identify original page references in this chunk
    page_numbers = []

    for item in selected_pages:
        marker = f"[PAGE {item['page_number']}]"

        if marker in chunk_text:
            page_numbers.append(item["page_number"])

    if not page_numbers:
        continue

    improved_chunks.append({
        "chunk_id": f"RBI_2017_CONTEXT_{len(improved_chunks)+1}",
        "source": "RBI_Customer_Protection_2017.pdf",
        "page_numbers": sorted(set(page_numbers)),
        "document_status": "historical",
        "document_scope": "customer_liability",
        "text": chunk_text
    })

# Save separately
output_path = Path(
    "documents/processed/rbi_context_chunks.json"
)

with open(output_path, "w", encoding="utf-8") as file:
    json.dump(improved_chunks, file, indent=4, ensure_ascii=False)

print("Context-preserving chunks created!")
print("Original pages:", len(selected_pages))
print("Improved chunks:", len(improved_chunks))
print("Saved to:", output_path)


Context-preserving chunks created!
Original pages: 4
Improved chunks: 5
Saved to: documents\processed\rbi_context_chunks.json


In [36]:
for item in improved_chunks:
    print("\n" + "=" * 65)
    print("CHUNK:", item["chunk_id"])
    print("SOURCE:", item["source"])
    print("PAGES:", item["page_numbers"])
    print("STATUS:", item["document_status"])
    print("TEXT LENGTH:", len(item["text"]))
    print("\nTEXT PREVIEW:")
    print(item["text"][:500])


CHUNK: RBI_2017_CONTEXT_1
SOURCE: RBI_Customer_Protection_2017.pdf
PAGES: [3]
STATUS: historical
TEXT LENGTH: 1200

TEXT PREVIEW:

[PAGE 3]
RBI,DBR,CO - 3 - continuation sheet communication systems used by banks to send alerts and receive their responses thereto must record the time and date of delivery of the message and receipt of customer’s response, if any, to them. This shall be important in determining the extent of a customer’s liability. The banks may not offer facility of electronic transactions, other than ATM cash withdrawals, to customers who do not provide mobile numbers to the bank. On receipt of report of an

CHUNK: RBI_2017_CONTEXT_2
SOURCE: RBI_Customer_Protection_2017.pdf
PAGES: [4]
STATUS: historical
TEXT LENGTH: 1200

TEXT PREVIEW:
reported by the customer). (ii) Third party breach where the deficiency lies neither with the bank nor with the customer but lies elsewhere in the system, and the customer notifies the bank within three working days of receiving the comm

In [37]:

import json
from pathlib import Path

# Collect the four original pages
liability_pages = sorted(
    [
        chunk for chunk in chunks
        if chunk["source"] == "RBI_Customer_Protection_2017.pdf"
        and chunk["page_number"] in [3, 4, 5, 6]
    ],
    key=lambda x: x["page_number"]
)

# Preserve every complete page
liability_text = "\n\n".join(
    f"[SOURCE PAGE {item['page_number']}]\n{item['text']}"
    for item in liability_pages
)

# Create one complete context record
liability_context = {
    "chunk_id": "RBI_2017_LIABILITY_CONTEXT_P3_P6",
    "source": "RBI_Customer_Protection_2017.pdf",
    "page_numbers": [3, 4, 5, 6],
    "document_status": "historical",
    "document_scope": "customer_liability",
    "chunk_type": "connected_regulatory_section",
    "text": liability_text
}

# Save separately
output_path = Path(
    "documents/processed/rbi_liability_context.json"
)

with open(output_path, "w", encoding="utf-8") as file:
    json.dump(
        [liability_context],
        file,
        indent=4,
        ensure_ascii=False
    )

print("Complete liability context created!")
print("Source pages:", liability_context["page_numbers"])
print("Text length:", len(liability_text))
print("Contains Table 1:", "Table 1" in liability_text)
print("Contains Table 2:", "Table 2" in liability_text)
print("Saved:", output_path)


Complete liability context created!
Source pages: [3, 4, 5, 6]
Text length: 7814
Contains Table 1: True
Contains Table 2: True
Saved: documents\processed\rbi_liability_context.json


In [38]:
for keyword in ["Table 1", "Table 2"]:
    position = liability_text.find(keyword)

    print("\n" + "=" * 60)
    print(keyword)
    print("=" * 60)

    if position != -1:
        print(liability_text[position:position + 1500])
    else:
        print("Table not found")


Table 1
Table 1, whichever is lower. Table 1 Maximum Liability of a Customer under paragraph 7 (ii) Type of Account Maximum liability (₹) • BSBD Accounts 5,000 • All other SB accounts • Pre-paid Payment Instruments and Gift Cards • Current/ Cash Credit/ Overdraft Accounts of MSMEs • Current Accounts/ Cash Credit/ Overdraft Accounts of Individuals with annual average balance (during 365 days preceding the incidence of fraud) / limit up to Rs.25 lakh • Credit cards with limit up to Rs.5 lakh 10,000 • All other Current/ Cash Credit/ Overdraft Accounts • Credit cards with limit above Rs.5 lakh 25,000 Further, if the delay in reporting is beyond seven working days, the customer liability shall be determined as per the bank’s B oard approved policy. Banks shall provide the details of their policy in regard to customers’ liability formulated in pursuance of these directions at the time of opening the accounts. Banks s hall also display their approved policy in public domain for wider dissemi

In [39]:
import json
from pathlib import Path

# Extract Table 1 and its related explanation
table1_start = liability_text.find("Table 1 Maximum Liability")
table1_end = liability_text.find("[SOURCE PAGE 5]")

table1_text = liability_text[table1_start:table1_end].strip()

# Extract Table 2 and its reporting-time explanation
table2_start = liability_text.find("Table 2 Summary of Customer’s Liability")
table2_end = liability_text.find("Reversal Timeline")

table2_text = liability_text[table2_start:table2_end].strip()

table_chunks = [
    {
        "chunk_id": "RBI_2017_TABLE_1_P4",
        "source": "RBI_Customer_Protection_2017.pdf",
        "page_number": 4,
        "document_status": "historical",
        "document_scope": "customer_liability",
        "chunk_type": "complete_table",
        "text": table1_text
    },
    {
        "chunk_id": "RBI_2017_TABLE_2_P5",
        "source": "RBI_Customer_Protection_2017.pdf",
        "page_number": 5,
        "document_status": "historical",
        "document_scope": "customer_liability",
        "chunk_type": "complete_table",
        "text": table2_text
    }
]

# Save the table-preserving chunks
output_path = Path("documents/processed/rbi_table_chunks.json")

with open(output_path, "w", encoding="utf-8") as file:
    json.dump(table_chunks, file, indent=4, ensure_ascii=False)

for chunk in table_chunks:
    print("\nChunk:", chunk["chunk_id"])
    print("Characters:", len(chunk["text"]))
    print("Text preview:\n", chunk["text"][:500])

print("\nSaved:", output_path)


Chunk: RBI_2017_TABLE_1_P4
Characters: 1318
Text preview:
 Table 1 Maximum Liability of a Customer under paragraph 7 (ii) Type of Account Maximum liability (₹) • BSBD Accounts 5,000 • All other SB accounts • Pre-paid Payment Instruments and Gift Cards • Current/ Cash Credit/ Overdraft Accounts of MSMEs • Current Accounts/ Cash Credit/ Overdraft Accounts of Individuals with annual average balance (during 365 days preceding the incidence of fraud) / limit up to Rs.25 lakh • Credit cards with limit up to Rs.5 lakh 10,000 • All other Current/ Cash Credit/ O

Chunk: RBI_2017_TABLE_2_P5
Characters: 535
Text preview:
 Table 2 Summary of Customer’s Liability Time taken to report the fraudulent transaction from the date of receiving the communication Customer’s liability (₹) Within 3 working days Zero liability Within 4 to 7 working days The transaction value or the amount mentioned in Table 1, whichever is lower Beyond 7 working days As per bank’s Board approved policy The number of working d

In [40]:
print("Model maximum sequence length:", model.max_seq_length)

for chunk in table_chunks:
    tokens = model.tokenizer(
        chunk["text"],
        truncation=False,
        add_special_tokens=True
    )["input_ids"]

    print("\nChunk:", chunk["chunk_id"])
    print("Total tokens:", len(tokens))
    print("Model limit:", model.max_seq_length)

    if len(tokens) > model.max_seq_length:
        print("WARNING: This chunk may be truncated during embedding.")
    else:
        print("OK: This chunk fits within the model limit.")

Model maximum sequence length: 256


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (284 > 256). Running this sequence through the model will result in indexing errors



Chunk: RBI_2017_TABLE_1_P4
Total tokens: 284
Model limit: 256

Chunk: RBI_2017_TABLE_2_P5
Total tokens: 101
Model limit: 256
OK: This chunk fits within the model limit.


In [41]:
import re
import json
from pathlib import Path

table1 = table_chunks[0]["text"]

# Split at the account-category bullet points
parts = re.split(r"(?=• )", table1)

print("Number of sections:", len(parts))

for i, part in enumerate(parts):
    if part.strip():
        token_count = len(
            model.tokenizer(
                part,
                truncation=False,
                add_special_tokens=True
            )["input_ids"]
        )

        print(f"\nSection {i + 1}")
        print("Characters:", len(part))
        print("Tokens:", token_count)
        print(part[:300])

Number of sections: 9

Section 1
Characters: 101
Tokens: 23
Table 1 Maximum Liability of a Customer under paragraph 7 (ii) Type of Account Maximum liability (₹) 

Section 2
Characters: 22
Tokens: 10
• BSBD Accounts 5,000 

Section 3
Characters: 24
Tokens: 7
• All other SB accounts 

Section 4
Characters: 46
Tokens: 11
• Pre-paid Payment Instruments and Gift Cards 

Section 5
Characters: 52
Tokens: 15
• Current/ Cash Credit/ Overdraft Accounts of MSMEs 

Section 6
Characters: 171
Tokens: 38
• Current Accounts/ Cash Credit/ Overdraft Accounts of Individuals with annual average balance (during 365 days preceding the incidence of fraud) / limit up to Rs.25 lakh 

Section 7
Characters: 49
Tokens: 17
• Credit cards with limit up to Rs.5 lakh 10,000 

Section 8
Characters: 53
Tokens: 14
• All other Current/ Cash Credit/ Overdraft Accounts 

Section 9
Characters: 800
Tokens: 165
• Credit cards with limit above Rs.5 lakh 25,000 Further, if the delay in reporting is beyond seven working days, th

In [42]:
table1_chunks = [
    {
        "chunk_id": "RBI_2017_TABLE_1_P4_A",
        "source": "RBI_Customer_Protection_2017.pdf",
        "page_number": 4,
        "document_status": "historical",
        "document_scope": "customer_liability",
        "chunk_type": "table_section",
        "text": """
Table 1: Maximum Liability of a Customer under paragraph 7(ii).

BSBD Accounts: Maximum liability ₹5,000.

The following account categories have a maximum liability of ₹10,000:
- All other SB accounts
- Pre-paid Payment Instruments and Gift Cards
- Current/Cash Credit/Overdraft Accounts of MSMEs
- Current Accounts/Cash Credit/Overdraft Accounts of Individuals
  with annual average balance or limit up to ₹25 lakh
- Credit cards with limit up to ₹5 lakh
"""
    },
    {
        "chunk_id": "RBI_2017_TABLE_1_P4_B",
        "source": "RBI_Customer_Protection_2017.pdf",
        "page_number": 4,
        "document_status": "historical",
        "document_scope": "customer_liability",
        "chunk_type": "table_section",
        "text": """
Table 1: Maximum Liability of a Customer under paragraph 7(ii).

The following account categories have a maximum liability of ₹25,000:
- All other Current/Cash Credit/Overdraft Accounts
- Credit cards with limit above ₹5 lakh
"""
    }
]

# Check token limits
for chunk in table1_chunks:
    count = len(
        model.tokenizer(
            chunk["text"],
            truncation=False
        )["input_ids"]
    )
    print(chunk["chunk_id"], ":", count, "tokens")

RBI_2017_TABLE_1_P4_A : 106 tokens
RBI_2017_TABLE_1_P4_B : 53 tokens


In [43]:
from pypdf import PdfReader
from pathlib import Path

pdf_path = Path(
    "documents/rbi/RBI_Customer_Protection_2017.pdf"
)

reader = PdfReader(str(pdf_path))

page4 = reader.pages[3]

layout_text = page4.extract_text(extraction_mode="layout")

print(layout_text)

           RBI,DBR,CO                              - 4 -                                        continuation sheet

             part of the customer in notifying the bank of such a transaction, the per
             transaction liability of the customer shall be limited to the transaction value
             or the amount mentioned in Table 1, whichever is lower.


                                                    Table 1
                       Maximum Liability of a Customer under paragraph 7 (ii)
                                Type of Account                                      Maximum
                                                                                    liability (₹)
          •   BSBD Accounts                                                                  5,000
          •   All other SB accounts
          •     Pre-paid Payment Instruments and Gift Cards
          •     Current/  Cash Credit/         Overdraft  Accounts of
              MSMEs
          •    Curr

In [44]:
table1_chunks = [
    {
        "chunk_id": "RBI_2017_TABLE_1_P4_A",
        "source": "RBI_Customer_Protection_2017.pdf",
        "page_number": 4,
        "document_status": "historical",
        "document_scope": "customer_liability",
        "chunk_type": "verified_table_section",
        "text": """
Historical RBI Customer Protection Circular, 2017.
Table 1: Maximum Liability of a Customer under paragraph 7(ii).

Maximum liability ₹5,000:
- BSBD Accounts.

Maximum liability ₹10,000:
- All other SB accounts.
- Pre-paid Payment Instruments and Gift Cards.
- Current/Cash Credit/Overdraft Accounts of MSMEs.
- Current Accounts/Cash Credit/Overdraft Accounts of Individuals
  with annual average balance or limit up to ₹25 lakh.
- Credit cards with limit up to ₹5 lakh.
"""
    },
    {
        "chunk_id": "RBI_2017_TABLE_1_P4_B",
        "source": "RBI_Customer_Protection_2017.pdf",
        "page_number": 4,
        "document_status": "historical",
        "document_scope": "customer_liability",
        "chunk_type": "verified_table_section",
        "text": """
Historical RBI Customer Protection Circular, 2017.
Table 1: Maximum Liability of a Customer under paragraph 7(ii).

Maximum liability ₹25,000:
- All other Current/Cash Credit/Overdraft Accounts.
- Credit cards with limit above ₹5 lakh.
"""
    }
]

# Combine with the previously saved Table 2
final_table_chunks = table1_chunks + [table_chunks[1]]

output_path = Path(
    "documents/processed/rbi_verified_table_chunks.json"
)

with open(output_path, "w", encoding="utf-8") as file:
    json.dump(
        final_table_chunks,
        file,
        indent=4,
        ensure_ascii=False
    )

# Verify token limits
for chunk in final_table_chunks:
    token_count = len(
        model.tokenizer(
            chunk["text"],
            truncation=False
        )["input_ids"]
    )

    print(chunk["chunk_id"], "| Tokens:", token_count)

print("\nVerified table chunks saved successfully!")

RBI_2017_TABLE_1_P4_A | Tokens: 113
RBI_2017_TABLE_1_P4_B | Tokens: 56
RBI_2017_TABLE_2_P5 | Tokens: 101

Verified table chunks saved successfully!


In [45]:
import json

# Load the final verified chunks
with open(
    "documents/processed/rbi_verified_table_chunks.json",
    "r",
    encoding="utf-8"
) as file:
    final_table_chunks = json.load(file)

# Prepare data for ChromaDB
ids = [chunk["chunk_id"] for chunk in final_table_chunks]
documents = [chunk["text"] for chunk in final_table_chunks]

metadatas = [
    {
        "source": chunk["source"],
        "page_number": chunk["page_number"],
        "document_status": chunk["document_status"],
        "document_scope": chunk["document_scope"],
        "chunk_type": chunk["chunk_type"]
    }
    for chunk in final_table_chunks
]

# Generate embeddings
table_embeddings = model.encode(
    documents,
    show_progress_bar=True
).tolist()

# Insert or update records without creating duplicate IDs
collection.upsert(
    ids=ids,
    documents=documents,
    embeddings=table_embeddings,
    metadatas=metadatas
)

print("Verified table chunks indexed successfully!")
print("Total ChromaDB records:", collection.count())

Batches: 100%|██████████| 1/1 [00:00<00:00, 10.08it/s]

Verified table chunks indexed successfully!
Total ChromaDB records: 36


In [46]:
test_queries = [
    "What is the maximum customer liability for BSBD accounts?",
    "What is the maximum liability for credit cards with limit above Rs. 5 lakh?",
    "What is the customer's liability when a fraudulent transaction is reported within 3 working days?",
    "What happens when the customer reports the fraudulent transaction after 7 working days?"
]

for question in test_queries:
    print("\n" + "=" * 75)
    print("QUESTION:", question)
    print("=" * 75)

    results = collection.query(
        query_embeddings=[
            model.encode(question).tolist()
        ],
        n_results=3,
        where={"document_status": "historical"}
    )

    for document, metadata, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        print(
            "\nSource:", metadata["source"],
            "| Page:", metadata["page_number"],
            "| Type:", metadata.get("chunk_type", "original_chunk"),
            "| Distance:", round(distance, 4)
        )
        print("Retrieved text:", document[:500])


QUESTION: What is the maximum customer liability for BSBD accounts?

Source: RBI_Customer_Protection_2017.pdf | Page: 4 | Type: verified_table_section | Distance: 0.5087
Retrieved text: 
Historical RBI Customer Protection Circular, 2017.
Table 1: Maximum Liability of a Customer under paragraph 7(ii).

Maximum liability ₹5,000:
- BSBD Accounts.

Maximum liability ₹10,000:
- All other SB accounts.
- Pre-paid Payment Instruments and Gift Cards.
- Current/Cash Credit/Overdraft Accounts of MSMEs.
- Current Accounts/Cash Credit/Overdraft Accounts of Individuals
  with annual average balance or limit up to ₹25 lakh.
- Credit cards with limit up to ₹5 lakh.


Source: RBI_Customer_Protection_2017.pdf | Page: 4 | Type: original_chunk | Distance: 0.7232
Retrieved text: RBI,DBR,CO - 4 - continuation sheet part of the customer in notifying the bank of such a transaction, the per transaction liability of the customer shall be limited to the transaction value or the amount mentioned in Table 1, whic

## 5. Regulatory retrieval tools


In [47]:
%%writefile src/regulatory_tools.py

import chromadb
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

client = chromadb.PersistentClient(path="documents/chroma_db")
collection = client.get_collection(name="rbi_regulations")


def format_evidence(results):
    evidence = []

    for i, document in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]

        evidence.append({
            "source": metadata["source"],
            "page": metadata["page_number"],
            "status": metadata.get("document_status", "unknown"),
            "scope": metadata.get("document_scope", "unknown"),
            "chunk_type": metadata.get("chunk_type", "original_chunk"),
            "text": document
        })

    return evidence


def search_regulations(question, top_k=5):
    embedding = model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[embedding],
        n_results=top_k
    )

    return format_evidence(results)


def retrieve_by_status(status, top_k=10):
    results = collection.get(
        where={"document_status": status},
        limit=top_k,
        include=["documents", "metadatas"]
    )

    evidence = []

    for document, metadata in zip(
        results["documents"],
        results["metadatas"]
    ):
        evidence.append({
            "source": metadata["source"],
            "page": metadata["page_number"],
            "status": metadata.get("document_status", "unknown"),
            "scope": metadata.get("document_scope", "unknown"),
            "chunk_type": metadata.get("chunk_type", "original_chunk"),
            "text": document
        })

    return evidence


def retrieve_page_evidence(source, page_number):
    results = collection.get(
        where={
            "$and": [
                {"source": source},
                {"page_number": page_number}
            ]
        },
        include=["documents", "metadatas"]
    )

    evidence = []

    for document, metadata in zip(
        results["documents"],
        results["metadatas"]
    ):
        evidence.append({
            "source": metadata["source"],
            "page": metadata["page_number"],
            "status": metadata.get("document_status", "unknown"),
            "scope": metadata.get("document_scope", "unknown"),
            "chunk_type": metadata.get("chunk_type", "original_chunk"),
            "text": document
        })

    return evidence


print("Regulatory retrieval tools updated successfully!")

Overwriting src/regulatory_tools.py


In [48]:
import importlib
import src.regulatory_tools as regulatory_tools

importlib.reload(regulatory_tools)

evidence = regulatory_tools.search_regulations(
    "What is the maximum liability for BSBD accounts?",
    top_k=3
)

for item in evidence:
    print("\nSource:", item["source"])
    print("Page:", item["page"])
    print("Status:", item["status"])
    print("Scope:", item["scope"])
    print("Chunk type:", item["chunk_type"])
    print("Evidence:", item["text"][:250])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1071.03it/s]


Regulatory retrieval tools updated successfully!


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1122.90it/s]


Regulatory retrieval tools updated successfully!

Source: RBI_Customer_Protection_2017.pdf
Page: 4
Status: historical
Scope: customer_liability
Chunk type: verified_table_section
Evidence: 
Historical RBI Customer Protection Circular, 2017.
Table 1: Maximum Liability of a Customer under paragraph 7(ii).

Maximum liability ₹5,000:
- BSBD Accounts.

Maximum liability ₹10,000:
- All other SB accounts.
- Pre-paid Payment Instruments and Gi

Source: RBI_Customer_Protection_2017.pdf
Page: 4
Status: historical
Scope: customer_liability
Chunk type: original_chunk
Evidence: RBI,DBR,CO - 4 - continuation sheet part of the customer in notifying the bank of such a transaction, the per transaction liability of the customer shall be limited to the transaction value or the amount mentioned in Table 1, whichever is lower. Tabl

Source: RBI_Customer_Protection_2017.pdf
Page: 4
Status: historical
Scope: customer_liability
Chunk type: verified_table_section
Evidence: 
Historical RBI Customer Protection C

In [49]:
import importlib
import src.regulatory_tools as regulatory_tools

importlib.reload(regulatory_tools)

questions = [
    "What is the maximum liability for BSBD accounts?",
    "What is the liability if a fraudulent transaction is reported within 3 working days?"
]

for question in questions:
    print("\nQUESTION:", question)

    evidence = regulatory_tools.search_regulations(
        question,
        top_k=3
    )

    for item in evidence:
        print(
            "\nSource:", item["source"],
            "| Page:", item["page"],
            "| Status:", item["status"]
        )
        print("Evidence:", item["text"][:400])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1339.04it/s]


Regulatory retrieval tools updated successfully!

QUESTION: What is the maximum liability for BSBD accounts?

Source: RBI_Customer_Protection_2017.pdf | Page: 4 | Status: historical
Evidence: 
Historical RBI Customer Protection Circular, 2017.
Table 1: Maximum Liability of a Customer under paragraph 7(ii).

Maximum liability ₹5,000:
- BSBD Accounts.

Maximum liability ₹10,000:
- All other SB accounts.
- Pre-paid Payment Instruments and Gift Cards.
- Current/Cash Credit/Overdraft Accounts of MSMEs.
- Current Accounts/Cash Credit/Overdraft Accounts of Individuals
  with annual average ba

Source: RBI_Customer_Protection_2017.pdf | Page: 4 | Status: historical
Evidence: RBI,DBR,CO - 4 - continuation sheet part of the customer in notifying the bank of such a transaction, the per transaction liability of the customer shall be limited to the transaction value or the amount mentioned in Table 1, whichever is lower. Table 1 Maximum Liability of a Customer under paragraph 7 (ii) Type of Account

In [50]:
historical = regulatory_tools.retrieve_by_status(
    "historical",
    top_k=10
)

current = regulatory_tools.retrieve_by_status(
    "effective_from_2026",
    top_k=10
)

print("Historical records retrieved:", len(historical))
print("2026 effective records retrieved:", len(current))

print("\nHistorical document names:")
for item in historical:
    print(item["source"], "| Page:", item["page"])

print("\n2026 document names:")
for item in current:
    print(item["source"], "| Page:", item["page"])

Historical records retrieved: 10
2026 effective records retrieved: 10

Historical document names:
RBI_Customer_Protection_2017.pdf | Page: 1
RBI_Customer_Protection_2017.pdf | Page: 2
RBI_Customer_Protection_2017.pdf | Page: 3
RBI_Customer_Protection_2017.pdf | Page: 4
RBI_Customer_Protection_2017.pdf | Page: 5
RBI_Customer_Protection_2017.pdf | Page: 6
RBI_Customer_Protection_2017.pdf | Page: 7
RBI_Customer_Protection_2017.pdf | Page: 8
RBI_Customer_Protection_2017.pdf | Page: 9
RBI_Customer_Protection_2017.pdf | Page: 4

2026 document names:
RBI_Integrated_Ombudsman_Scheme_2026.pdf | Page: 1
RBI_Integrated_Ombudsman_Scheme_2026.pdf | Page: 2
RBI_Integrated_Ombudsman_Scheme_2026.pdf | Page: 3
RBI_Integrated_Ombudsman_Scheme_2026.pdf | Page: 4
RBI_Integrated_Ombudsman_Scheme_2026.pdf | Page: 5
RBI_Integrated_Ombudsman_Scheme_2026.pdf | Page: 6
RBI_Integrated_Ombudsman_Scheme_2026.pdf | Page: 7
RBI_Integrated_Ombudsman_Scheme_2026.pdf | Page: 8
RBI_Integrated_Ombudsman_Scheme_2026.pdf |

In [51]:
all_records = collection.get(
    include=["metadatas"]
)

status_counts = {}

for metadata in all_records["metadatas"]:
    status = metadata.get("document_status", "unknown")
    status_counts[status] = status_counts.get(status, 0) + 1

print("Total ChromaDB records:", collection.count())

for status, count in status_counts.items():
    print(status, ":", count)

Total ChromaDB records: 36
historical : 12
effective_from_2026 : 24


## 6. Five-category fraud classification


In [52]:
%%writefile src/fraud_classifier.py

import re

FRAUD_CATEGORIES = [
    "Card-not-present fraud",
    "Skimming",
    "UPI social engineering",
    "Merchant dispute",
    "Duplicate charge"
]

patterns = {
    "Duplicate charge": [
        r"\bduplicate\b",
        r"\bcharged twice\b",
        r"\bdebited twice\b",
        r"\btwo charges\b",
        r"\bsame transaction twice\b",
        r"\btransaction\b.*\bappeared twice\b",
        r"\bappeared twice\b"
    ],

    "Skimming": [
        r"\bskimming\b",
        r"\bskimmer\b",
        r"\bcloned card\b",
        r"\bcard cloned\b",
        r"\bcard was cloned\b",
        r"\bcloned at an atm\b",
        r"\bcard data copied\b",
        r"\bcompromised atm\b"
    ],

    "UPI social engineering": [
        r"\bupi\b",
        r"\bqr code\b",
        r"\bcollect request\b",
        r"\bimpersonat\w*\b",
        r"\bpretend\w*\b",
        r"\botp shared\b",
        r"\botp disclosed\b"
    ],

    "Card-not-present fraud": [
        r"\bonline card\b",
        r"\bcard not present\b",
        r"\bwithout physical card\b",
        r"\bunauthori[sz]ed online card\b",
        r"\bunauthori[sz]ed online purchase\b",
        r"\bunknown online purchase\b",
        r"\bused online\b",
        r"\bcard details\b.*\bused\b",
        r"\bcard details\b.*\bonline\b",
        r"\bonline transaction\b"
    ],

    "Merchant dispute": [
        r"\brefund\b",
        r"\bproduct not received\b",
        r"\bnever received my product\b",
        r"\bnever received my order\b",
        r"\bnever received the order\b",
        r"\border not received\b",
        r"\bservice not received\b",
        r"\bservice\b.*\bnot received\b",
        r"\bwrong product\b",
        r"\bmerchant dispute\b",
        r"\bcharged but order cancelled\b",
        r"\border\b.*\bcancelled\b"
    ]
}


def classify_fraud(description):
    text = description.lower()
    scores = {}

    for category, keywords in patterns.items():
        scores[category] = sum(
            1 for pattern in keywords
            if re.search(pattern, text)
        )

    highest_score = max(scores.values())

    if highest_score == 0:
        return {
            "category": "Needs manual review",
            "confidence": "Low",
            "reason": "No clear matching indicators found."
        }

    winners = [
        category for category, score in scores.items()
        if score == highest_score
    ]

    if len(winners) > 1:
        return {
            "category": "Needs manual review",
            "confidence": "Low",
            "reason": "Multiple categories have equal matching scores."
        }

    return {
        "category": winners[0],
        "confidence": "Rule-based preliminary match",
        "reason": f"Matched indicators: {highest_score}"
    }


print("Fraud classifier updated successfully!")

Overwriting src/fraud_classifier.py


In [53]:
from src.fraud_classifier import classify_fraud

test_cases = [
    {
        "expected": "Card-not-present fraud",
        "description": "Someone used my card details to make an unauthorized online purchase."
    },
    {
        "expected": "Skimming",
        "description": "My card was cloned at an ATM using a skimmer."
    },
    {
        "expected": "UPI social engineering",
        "description": "A caller pretending to be my bank tricked me into sharing my UPI OTP."
    },
    {
        "expected": "Merchant dispute",
        "description": "I paid for an order but never received my product."
    },
    {
        "expected": "Duplicate charge",
        "description": "The same transaction was charged twice."
    }
]

results = []

for case in test_cases:
    prediction = classify_fraud(case["description"])

    results.append({
        "expected": case["expected"],
        "predicted": prediction["category"],
        "passed": prediction["category"] == case["expected"]
    })

for result in results:
    print(result)

passed = sum(r["passed"] for r in results)

print("\nRESULT:", passed, "/", len(results), "passed")

Fraud classifier updated successfully!
{'expected': 'Card-not-present fraud', 'predicted': 'Card-not-present fraud', 'passed': True}
{'expected': 'Skimming', 'predicted': 'Skimming', 'passed': True}
{'expected': 'UPI social engineering', 'predicted': 'UPI social engineering', 'passed': True}
{'expected': 'Merchant dispute', 'predicted': 'Merchant dispute', 'passed': True}
{'expected': 'Duplicate charge', 'predicted': 'Duplicate charge', 'passed': True}

RESULT: 5 / 5 passed


In [54]:
import json
from pathlib import Path
from datetime import datetime

fraud_report = {
    "test_name": "Five Category Fraud Classification",
    "status": "PASSED",
    "total_test_cases": len(results),
    "passed": passed,
    "failed": len(results) - passed,
    "accuracy_on_test_cases": round(
        passed / len(results) * 100, 2
    ),
    "classifier_type": "Rule-based",
    "results": results,
    "test_date": datetime.now().isoformat()
}

with open(
    "evaluation/fraud_five_category_report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(fraud_report, f, indent=4)

print("Fraud evaluation report saved!")
print(json.dumps(fraud_report, indent=4))

Fraud evaluation report saved!
{
    "test_name": "Five Category Fraud Classification",
    "status": "PASSED",
    "total_test_cases": 5,
    "passed": 5,
    "failed": 0,
    "accuracy_on_test_cases": 100.0,
    "classifier_type": "Rule-based",
    "results": [
        {
            "expected": "Card-not-present fraud",
            "predicted": "Card-not-present fraud",
            "passed": true
        },
        {
            "expected": "Skimming",
            "predicted": "Skimming",
            "passed": true
        },
        {
            "expected": "UPI social engineering",
            "predicted": "UPI social engineering",
            "passed": true
        },
        {
            "expected": "Merchant dispute",
            "predicted": "Merchant dispute",
            "passed": true
        },
        {
            "expected": "Duplicate charge",
            "predicted": "Duplicate charge",
            "passed": true
        }
    ],
    "test_date": "2026-11-11T09:39:42

## 7. Gemini-powered RAG answer generation


In [55]:
import os
from getpass import getpass

os.environ["GEMINI_API_KEY"] = getpass(
    "Enter your Gemini API key: "
)

print("Gemini API key loaded securely!")

Enter your Gemini API key:  ········


Gemini API key loaded securely!


In [56]:
from pathlib import Path

code = r'''
import os
import requests

GEMINI_URL = (
    "https://generativelanguage.googleapis.com/v1beta/"
    "models/gemini-3.8-flash:generateContent"
)

def generate_rag_answer(question, evidence, timeout=90):
    evidence_text = ""

    for item in evidence[:3]:
        evidence_text += f"""
Source: {item['source']}
Page: {item['page']}
Status: {item['status']}
Scope: {item['scope']}
Evidence: {item['text'][:1000]}
--------------------
"""

    prompt = f"""
You are DisputeAI's RBI regulatory assistant.

Answer using only the evidence provided.

Question:
{question}

Evidence:
{evidence_text}

Instructions:
- Cite source names and page numbers.
- Never present withdrawn 2017 rules as current law.
- Distinguish historical from current evidence.
- Do not invent RBI rules or bank policies.
- Identify missing current regulations.
- Recommend human review if liability cannot be determined.
"""

    response = requests.post(
        GEMINI_URL,
        params={"key": os.environ["GEMINI_API_KEY"]},
        json={
            "contents": [
                {"parts": [{"text": prompt}]}
            ]
        },
        timeout=timeout
    )

    response.raise_for_status()
    result = response.json()

    return result["candidates"][0]["content"]["parts"][0]["text"]
'''

Path("src/gemini_rag.py").write_text(code, encoding="utf-8")

print("Gemini RAG module saved successfully!")

Gemini RAG module saved successfully!


In [57]:
from src.gemini_rag import generate_rag_answer
from src.regulatory_tools import search_regulations

question = """
I noticed an unauthorized UPI transaction.
Someone called pretending to be from my bank,
and I shared my OTP. What should I do?
"""

evidence = search_regulations(question, top_k=5)

print("Evidence retrieved:", len(evidence))

answer = generate_rag_answer(
    question=question,
    evidence=evidence,
    timeout=90
)

print("\nDISPUTEAI FINAL ANSWER")
print("=" * 60)
print(answer)

Evidence retrieved: 5

DISPUTEAI FINAL ANSWER
Based on the provided evidence, here is what you should do and what you need to know:

### 1. Immediate Action
* **Report the unauthorized transaction to your bank immediately:** Under historical guidelines in *RBI_Customer_Protection_2017.pdf* (Page 3), upon receipt of a report of an unauthorized transaction from a customer, banks are required to take immediate steps to prevent further unauthorized transactions in the account. Reporting mechanisms are also referenced in *RBI_Customer_Protection_2017.pdf* (Page 2).

---

### 2. Historical Regulatory Context vs. Current Law
* **Status of Provided Evidence:** The only evidence provided comes from *RBI_Customer_Protection_2017.pdf* (Pages 2, 3, and 6), which carries the status of **historical** and **cannot be presented or relied upon as current law**. 
* **Customer Liability & Negligence:** 
  * Under *RBI_Customer_Protection_2017.pdf* (Page 3), zero liability applies in cases of contributory

In [58]:
import json
from pathlib import Path
from datetime import datetime

report = {
    "test_name": "Gemini RAG Integration",
    "status": "PASSED",
    "model": "gemini-3.8-flash",
    "retrieved_evidence_count": len(evidence),
    "answer_generated": bool(answer),
    "historical_warning_present": "historical" in answer.lower(),
    "human_review_recommended": "human review" in answer.lower(),
    "test_date": datetime.now().isoformat()
}

Path("evaluation").mkdir(exist_ok=True)

with open(
    "evaluation/gemini_rag_report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(report, f, indent=4)

print(json.dumps(report, indent=4))
print("\nRAG evaluation report saved!")

{
    "test_name": "Gemini RAG Integration",
    "status": "PASSED",
    "model": "gemini-3.8-flash",
    "retrieved_evidence_count": 5,
    "answer_generated": true,
    "historical_warning_present": true,
    "human_review_recommended": true,
    "test_date": "2026-11-11T09:41:05.639166"
}

RAG evaluation report saved!


In [59]:
print("Generated answer:")
print(answer)

print("\nRetrieved evidence sources:")

for i, item in enumerate(evidence, 1):
    print(f"\nEvidence {i}")
    print("Source:", item["source"])
    print("Page:", item["page"])
    print("Status:", item["status"])
    print("Scope:", item["scope"])

Generated answer:
Based on the provided evidence, here is what you should do and what you need to know:

### 1. Immediate Action
* **Report the unauthorized transaction to your bank immediately:** Under historical guidelines in *RBI_Customer_Protection_2017.pdf* (Page 3), upon receipt of a report of an unauthorized transaction from a customer, banks are required to take immediate steps to prevent further unauthorized transactions in the account. Reporting mechanisms are also referenced in *RBI_Customer_Protection_2017.pdf* (Page 2).

---

### 2. Historical Regulatory Context vs. Current Law
* **Status of Provided Evidence:** The only evidence provided comes from *RBI_Customer_Protection_2017.pdf* (Pages 2, 3, and 6), which carries the status of **historical** and **cannot be presented or relied upon as current law**. 
* **Customer Liability & Negligence:** 
  * Under *RBI_Customer_Protection_2017.pdf* (Page 3), zero liability applies in cases of contributory fraud/negligence/deficiency

In [60]:
from pypdf import PdfReader

pdf_path = "documents/rbi/RBI_Customer_Protection_2017.pdf"

reader = PdfReader(pdf_path)

for page_number in [3, 6]:
    print("\n" + "=" * 60)
    print("ORIGINAL PDF PAGE:", page_number)
    print("=" * 60)
    print(reader.pages[page_number - 1].extract_text()[:3500])


ORIGINAL PDF PAGE: 3
RBI,DBR,CO - 3 -                                        continuation sheet 
 
communication systems used by banks to send alerts and receive their responses 
thereto must record the time and date of delivery of the message and receipt of 
customer’s response, if any, to them. This shall be important in determining the 
extent of a customer’s liability. The banks may not offer facility of electronic 
transactions, other than ATM cash withdrawals, to customers who do not provide 
mobile numbers to the bank. On receipt of report of an unauthorised transaction from 
the customer, banks must take immediate steps to prevent further unauthorised 
transactions in the account. 
 
Limited Liability of a Customer 
(a) Zero Liability of a Customer 
6. A customer’s entitlement to zero liability shall arise where the unauthorised 
transaction occurs in the following events: 
(i) Contributory fraud/ negligence/ deficiency on the part of the bank 
(irrespective of whether or not 

In [61]:
import json
from pathlib import Path
from datetime import datetime

citation_report = {
    "test_name": "RAG Citation Verification",
    "status": "PASSED",
    "verified_citations": [
        {
            "source": "RBI_Customer_Protection_2017.pdf",
            "page": 3,
            "claim": "Banks must take immediate steps to prevent further unauthorized transactions",
            "verified_against_original_pdf": True
        },
        {
            "source": "RBI_Customer_Protection_2017.pdf",
            "page": 6,
            "claim": "Banks must formulate Board-approved customer protection policies",
            "verified_against_original_pdf": True
        }
    ],
    "historical_document_warning": True,
    "current_customer_liability_rules_available": False,
    "test_date": datetime.now().isoformat()
}

with open(
    "evaluation/citation_verification_report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(citation_report, f, indent=4)

print("Citation verification report saved successfully!")
print(json.dumps(citation_report, indent=4))

Citation verification report saved successfully!
{
    "test_name": "RAG Citation Verification",
    "status": "PASSED",
    "verified_citations": [
        {
            "source": "RBI_Customer_Protection_2017.pdf",
            "page": 3,
            "claim": "Banks must take immediate steps to prevent further unauthorized transactions",
            "verified_against_original_pdf": true
        },
        {
            "source": "RBI_Customer_Protection_2017.pdf",
            "page": 6,
            "claim": "Banks must formulate Board-approved customer protection policies",
            "verified_against_original_pdf": true
        }
    ],
    "historical_document_warning": true,
    "current_customer_liability_rules_available": false,
    "test_date": "2026-11-11T09:41:05.907930"
}


## 8. LangGraph investigation workflow


In [62]:
%%writefile src/investigation_agents.py

from src.fraud_classifier import classify_fraud
from src.regulatory_tools import search_regulations


def fraud_investigation_agent(state):
    """Agent 1: Classifies the customer's dispute."""

    result = classify_fraud(state["dispute"])

    return {
        "fraud_category": result["category"],
        "fraud_reason": result["reason"]
    }


def regulatory_compliance_agent(state):
    """Agent 2: Retrieves regulatory evidence."""

    question = (
        state["dispute"]
        + " "
        + state["fraud_category"]
    )

    evidence = search_regulations(question, top_k=5)

    return {
        "regulatory_evidence": evidence
    }


print("Both investigation agents created successfully!")

Overwriting src/investigation_agents.py


In [63]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, END

from src.investigation_agents import (
    fraud_investigation_agent,
    regulatory_compliance_agent
)

from src.fraud_classifier import classify_fraud

print("LangGraph imports successful!")
print("Fraud classifier loaded!")
print("Regulatory agent loaded!")

Both investigation agents created successfully!
LangGraph imports successful!
Fraud classifier loaded!
Regulatory agent loaded!


In [64]:
class InvestigationState(TypedDict):
    dispute: str
    fraud_category: str
    fraud_reason: str
    regulatory_evidence: List[dict]
    investigation_summary: str

print("InvestigationState created successfully!")

InvestigationState created successfully!


In [65]:
def investigation_assessment_agent(state):
    dispute = state["dispute"].lower()
    evidence = state["regulatory_evidence"]

    findings = []

    if "otp" in dispute and ("shared" in dispute or "share" in dispute):
        findings.append(
            "Customer-related indicator: OTP was shared with another person."
        )

    if "impersonat" in dispute or "pretending" in dispute:
        findings.append(
            "Third-party indicator: Caller allegedly impersonated the bank."
        )

    historical_liability = [
        item for item in evidence
        if item.get("status") == "historical"
        and item.get("scope") == "customer_liability"
    ]

    current_complaint = [
        item for item in evidence
        if item.get("status") == "effective_from_2026"
        and item.get("scope") == "complaint_redressal"
    ]

    current_liability = [
        item for item in evidence
        if item.get("status") == "current"
        and item.get("scope") == "customer_liability"
    ]

    if historical_liability:
        findings.append(
            "Historical customer-liability provisions were retrieved. "
            "They must not be treated as current rules."
        )

    if current_complaint:
        findings.append(
            "Current complaint-redressal evidence was retrieved. "
            "It does not establish current customer-liability rules."
        )

    if not current_liability:
        findings.append(
            "Current customer-liability guidance was not found "
            "in the available indexed documents."
        )

    findings.append(
        "Final liability cannot be determined from the available evidence. "
        "Obtain applicable current RBI provisions and bank policy, "
        "then escalate for human review."
    )

    return {
        "investigation_summary": "\n".join(
            "- " + finding for finding in findings
        )
    }


workflow = StateGraph(InvestigationState)

workflow.add_node("fraud_agent", fraud_investigation_agent)
workflow.add_node("compliance_agent", regulatory_compliance_agent)
workflow.add_node("assessment_agent", investigation_assessment_agent)

workflow.set_entry_point("fraud_agent")

workflow.add_edge("fraud_agent", "compliance_agent")
workflow.add_edge("compliance_agent", "assessment_agent")
workflow.add_edge("assessment_agent", END)

app = workflow.compile()

print("LangGraph workflow compiled successfully!")

LangGraph workflow compiled successfully!


In [66]:
test_dispute = {
    "dispute": """
    Someone called pretending to be from my bank.
    They asked for my OTP, which I shared.
    Later, I noticed an unauthorized UPI transaction.
    """
}

result = app.invoke(test_dispute)

print("DISPUTEAI INVESTIGATION REPORT")
print("=" * 60)

print("\nFraud Category:")
print(result["fraud_category"])

print("\nFraud Reason:")
print(result["fraud_reason"])

print("\nRetrieved RBI Evidence:")
for item in result["regulatory_evidence"]:
    print(
        f"- {item['source']} | "
        f"Page {item['page']} | "
        f"Status: {item['status']}"
    )

print("\nInvestigation Summary:")
print(result["investigation_summary"])

DISPUTEAI INVESTIGATION REPORT

Fraud Category:
UPI social engineering

Fraud Reason:
Matched indicators: 2

Retrieved RBI Evidence:
- RBI_Customer_Protection_2017.pdf | Page 3 | Status: historical
- RBI_Customer_Protection_2017.pdf | Page 6 | Status: historical
- RBI_Customer_Protection_2017.pdf | Page 2 | Status: historical
- RBI_Integrated_Ombudsman_Scheme_2026.pdf | Page 21 | Status: effective_from_2026
- RBI_Customer_Protection_2017.pdf | Page 5 | Status: historical

Investigation Summary:
- Customer-related indicator: OTP was shared with another person.
- Third-party indicator: Caller allegedly impersonated the bank.
- Historical customer-liability provisions were retrieved. They must not be treated as current rules.
- Current complaint-redressal evidence was retrieved. It does not establish current customer-liability rules.
- Current customer-liability guidance was not found in the available indexed documents.
- Final liability cannot be determined from the available evidence. O

In [67]:
import json
from pathlib import Path
from datetime import datetime

langgraph_report = {
    "test_name": "LangGraph End-to-End Investigation",
    "status": "PASSED",
    "agents": [
        "Fraud Investigation Agent",
        "Regulatory Compliance Agent",
        "Investigation Assessment Agent"
    ],
    "fraud_category": result["fraud_category"],
    "expected_category": "UPI social engineering",
    "classification_passed": (
        result["fraud_category"] == "UPI social engineering"
    ),
    "evidence_records_retrieved": len(
        result["regulatory_evidence"]
    ),
    "historical_warning_present": (
        "must not be treated as current rules"
        in result["investigation_summary"]
    ),
    "missing_current_liability_detected": (
        "not found" in result["investigation_summary"]
    ),
    "human_review_recommended": (
        "human review" in result["investigation_summary"]
    ),
    "test_date": datetime.now().isoformat()
}

Path("evaluation").mkdir(exist_ok=True)

with open(
    "evaluation/langgraph_final_report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(langgraph_report, f, indent=4)

print(json.dumps(langgraph_report, indent=4))
print("\nLangGraph evaluation report saved successfully!")

{
    "test_name": "LangGraph End-to-End Investigation",
    "status": "PASSED",
    "agents": [
        "Fraud Investigation Agent",
        "Regulatory Compliance Agent",
        "Investigation Assessment Agent"
    ],
    "fraud_category": "UPI social engineering",
    "expected_category": "UPI social engineering",
    "classification_passed": true,
    "evidence_records_retrieved": 5,
    "historical_warning_present": true,
    "missing_current_liability_detected": true,
    "human_review_recommended": true,
    "test_date": "2026-11-11T09:41:08.392642"
}

LangGraph evaluation report saved successfully!


## 9. Final project and persistence checks


In [68]:
from pathlib import Path

folders = [
    "src",
    "documents/rbi",
    "documents/bank_policies",
    "documents/processed",
    "documents/metadata",
    "documents/chroma_db",
    "evaluation"
]

for folder in folders:
    path = Path(folder)
    print(f"\n{folder}: {'EXISTS' if path.exists() else 'MISSING'}")

    if path.exists() and folder in ["src", "evaluation"]:
        for file in path.iterdir():
            if file.is_file():
                print("  -", file.name)


src: EXISTS
  - circuit_breaker.py
  - document_indexer.py
  - fraud_classifier.py
  - gemini_rag.py
  - input_security.py
  - investigation_agents.py
  - investigation_prompts.py
  - output_security.py
  - rag_retriever.py
  - regulatory_tools.py
  - reliable_retrieval.py

documents/rbi: EXISTS

documents/bank_policies: EXISTS

documents/processed: EXISTS

documents/metadata: EXISTS

documents/chroma_db: EXISTS

evaluation: EXISTS
  - citation_verification_report.json
  - fraud_classifier_report.json
  - fraud_five_category_report.json
  - gemini_rag_report.json
  - langgraph_final_report.json
  - langgraph_integration_report.json
  - test_scenarios.py


In [69]:
from pathlib import Path

folders = [
    "documents/rbi",
    "documents/bank_policies",
    "documents/processed",
    "documents/metadata"
]

for folder in folders:
    print("\n" + "=" * 50)
    print(folder)
    print("=" * 50)

    files = list(Path(folder).glob("*"))

    if not files:
        print("Folder is empty")
    else:
        for file in files:
            if file.is_file():
                print(file.name, "-", file.stat().st_size, "bytes")


documents/rbi
RBI_Customer_Protection_2017.pdf - 621140 bytes
RBI_Integrated_Ombudsman_Scheme_2026.pdf - 260017 bytes

documents/bank_policies
Folder is empty

documents/processed
rbi_chunks.json - 64854 bytes
rbi_context_chunks.json - 7451 bytes
RBI_Customer_Protection_2017_cleaned.txt - 16279 bytes
RBI_Customer_Protection_2017_extracted.txt - 16925 bytes
RBI_Integrated_Ombudsman_Scheme_2026_cleaned.txt - 43321 bytes
RBI_Integrated_Ombudsman_Scheme_2026_extracted.txt - 44257 bytes
rbi_liability_context.json - 8268 bytes
rbi_table_chunks.json - 2474 bytes
rbi_verified_table_chunks.json - 2187 bytes

documents/metadata
rbi_cleaned_pages.json - 64834 bytes
rbi_extracted_pages.json - 64514 bytes


In [70]:
import chromadb

client_check = chromadb.PersistentClient(
    path="documents/chroma_db"
)

collection_check = client_check.get_collection(
    name="rbi_regulations"
)

print("ChromaDB persistence check")
print("--------------------------")
print("Collection:", collection_check.name)
print("Total indexed records:", collection_check.count())

ChromaDB persistence check
--------------------------
Collection: rbi_regulations
Total indexed records: 36
